# Egyptian Civil Code — Streamlit test app (Colab launcher)

Opens a web page where you ask the fine-tuned model in Arabic, see the article it used, and rate the answer. Ratings are saved to `validation_log.csv`.

**Runtime:** Runtime → Change runtime type → **GPU (T4)**.

**You upload 3 files:** `app.py`, `egyptian_civil_code_articles2.json`, `egyptian_lora_model.zip`.

> The link works only while this Colab session is running. Keep this tab open, and download `validation_log.csv` from the app's sidebar before you close it. Anyone who has the link can use the app.

## Step 1 — Install

In [ ]:
!pip install -q unsloth streamlit sentence-transformers rank_bm25 python-docx pymupdf

## Step 2 — Upload the files

In [ ]:
from google.colab import files
import os

print("Upload: app.py, egyptian_civil_code_articles2.json, egyptian_lora_model.zip")
files.upload()
for need in ["app.py", "egyptian_civil_code_articles2.json", "egyptian_lora_model.zip"]:
    assert os.path.exists(need), f"{need} is missing. Run this cell again and select it."
print("All files are here.")

## Step 3 — Start the app and get the public link

In [ ]:
import subprocess, time, re, os

# 1) the app
st_log = open("streamlit.log", "w")
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
                  "--server.fileWatcherType", "none", "--browser.gatherUsageStats", "false"],
                 stdout=st_log, stderr=st_log)

# 2) a free public tunnel (no account needed)
if not os.path.exists("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared
cf_log = open("cloudflared.log", "w")
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                 stdout=cf_log, stderr=cf_log)

url = None
for _ in range(60):
    time.sleep(2)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("cloudflared.log").read())
    if m:
        url = m.group(0); break
print("Open this link:", url or "not found yet. Run the next cell to see the logs.")
print("The first load takes about 2-3 minutes (the model and the retriever load once).")

In [ ]:
# Only if something went wrong: show the last lines of both logs.
print("--- streamlit.log ---");   print(open("streamlit.log").read()[-1500:])
print("--- cloudflared.log ---"); print(open("cloudflared.log").read()[-800:])

In [ ]:
# Run this before you close Colab: saves your ratings to your laptop.
from google.colab import files
import os
if os.path.exists("validation_log.csv"):
    files.download("validation_log.csv")
else:
    print("No ratings saved yet.")